In [9]:
import os
# חשפי את ה-GPU-ים (או רק "0" אם את רוצה כרטיס יחיד)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4"
# בטלי את המסכה שמגיעה מסביבת דוקר/קוברנטיס ומסתירה כרטיסים
os.environ.pop("NVIDIA_VISIBLE_DEVICES", None)

import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
print("count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print(subprocess.check_output(["nvidia-smi"], text=True).splitlines()[0])


CUDA available: True
count: 3
0 NVIDIA H100 PCIe
1 NVIDIA H100 PCIe
2 NVIDIA H100 PCIe
Sun Nov 16 07:15:50 2025       


#0) Mount + paths + helpers



In [2]:
from pathlib import Path
import os, shutil, json

# Local workspace
LOCAL_ROOT     = Path("")
LOCAL_DATA_DIR = LOCAL_ROOT / "data"
LOCAL_OUT_DIR  = LOCAL_ROOT / "outputs"
LOCAL_CFG_DIR  = LOCAL_ROOT / "axolotl_configs"
LOCAL_LOG_DIR  = LOCAL_ROOT / "logs"
for p in [LOCAL_DATA_DIR, LOCAL_OUT_DIR, LOCAL_CFG_DIR, LOCAL_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


print("Local root     :", LOCAL_ROOT)


Local root     : .


# Write Axolotl configs to local (separate caches + outputs)

In [3]:
from pathlib import Path
LOCAL_ROOT    = Path("") #Path("/content/ft_vs_rag_work") #DONE
LOCAL_CFG_DIR =  LOCAL_ROOT / "axolotl_configs"
LOCAL_OUT_DIR =  LOCAL_ROOT / "outputs"
LOCAL_DATA_DIR=  Path("data") #LOCAL_ROOT / "data" 
LOCAL_CFG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
SEQ_LEN    = 1760 #2048 #TODO
STAGE1_DIR = (LOCAL_OUT_DIR / "mistral_stage1")
STAGE2_DIR = (LOCAL_OUT_DIR / "mistral_stage2")

pretrain_file = (LOCAL_DATA_DIR / "docs_for_pretrain (1).jsonl") #parquet")
sft_file      = (LOCAL_DATA_DIR / "docs_qa_with_context_ALPACA.jsonl")

assert pretrain_file.exists(), f"missing {pretrain_file}"
assert sft_file.exists(),      f"missing {sft_file}"

STAGE1_MAX_STEPS = 2770 # 8300 # 4800 #4170  # TODO update

pretrain_yaml = f"""
base_model: {BASE_MODEL}
model_type: mistral
tokenizer_type: AutoTokenizer
trust_remote_code: true
max_steps: {STAGE1_MAX_STEPS}

###############
load_in_4bit: true
bnb_4bit_quant_type: nf4
bnb_4bit_compute_dtype: bfloat16
# torch_dtype: bfloat16
dtype: bfloat16
gradient_checkpointing: true
attn_implementation: flash_attention_2 #DONE
# attn_implementation: sdpa

sequence_len: {SEQ_LEN}
micro_batch_size: 64
gradient_accumulation_steps: 1
eval_micro_batch_size: 64

bf16: true
tf32: true

# DataLoader
# dataloader_num_workers: 0 #2
# dataloader_pin_memory: true

packing: true
pad_to_sequence_len: false

# deepspeed: axolotl_configs/ds_config_zero3.json #Done

###########

adapter: qlora

# --- Stage 1 (pretrain) ---
dataset_prepared_path: cache_pretrain
skip_prepare_dataset: false

debug_num_examples: 0

pretraining_dataset:
  - data_files:
      - {pretrain_file}
    path: "json" 
    # streaming: true
#pretraining_dataset:
#  - data_files:
#      - {pretrain_file} # updated
#    path: json # parquet 
#    streaming: false # updated
#    trust_remote_code: false # added
#    text_column: text
#    text_field: text
#    shuffle: true
#    seed: 42

output_dir: {STAGE1_DIR.as_posix()}

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
bias: "none"
task_type: "CAUSAL_LM"
lora_target_linear: true
adapters: lora

num_epochs: 1
save_steps: 0
evals_per_epoch: 0

optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 2e-4
warmup_steps: 100
logging_steps: 10

# preprocessing_num_workers: 1
"""

sft_yaml = f"""
base_model: {BASE_MODEL}
model_type: mistral
tokenizer_type: AutoTokenizer
trust_remote_code: true

###############
load_in_4bit: true
bnb_4bit_quant_type: nf4
bnb_4bit_compute_dtype: bfloat16
#torch_dtype: bfloat16
dtype: bfloat16
gradient_checkpointing: true
attn_implementation: flash_attention_2 #DONE
# attn_implementation: sdpa

micro_batch_size: 48
gradient_accumulation_steps: 1
eval_micro_batch_size: 48

bf16: true
tf32: true

# DataLoader
# dataloader_num_workers: 0 #2
# dataloader_pin_memory: true

packing: true
pad_to_sequence_len: false

# deepspeed: axolotl_configs/ds_config_zero3.json #DONE

###########

adapter: qlora
lora_path: {STAGE1_DIR.as_posix()}

datasets:
  - path: {sft_file}
    type: json
    type: alpaca
    prompt_template: alpaca


system_prompt: |
  You are a helpful assistant. Use the provided context when it helps,
  but you are not required to rely on it exclusively. Respond with your best knowledge and reasoning.

dataset_prepared_path: cache_sft
output_dir: {STAGE2_DIR.as_posix()}

sequence_len: {SEQ_LEN}

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
bias: "none"
task_type: "CAUSAL_LM"
lora_target_linear: true
adapters: lora

num_epochs: 3 #DONE

optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 1e-4
warmup_steps: 100
train_on_inputs: false
group_by_length: false
save_every: 0
evals_per_epoch: 0
logging_steps: 10

# preprocessing_num_workers: 8
"""

(LOCAL_CFG_DIR / "config_stage1_pretrain.yaml").write_text(pretrain_yaml)
(LOCAL_CFG_DIR / "config_stage2_sft.yaml").write_text(sft_yaml)

print("Configs written:\n -", LOCAL_CFG_DIR / "config_stage1_pretrain.yaml", "\n -", LOCAL_CFG_DIR / "config_stage2_sft.yaml")


Configs written:
 - axolotl_configs/config_stage1_pretrain.yaml 
 - axolotl_configs/config_stage2_sft.yaml


# Cell 6 — (Optional) Preprocess datasets

In [93]:
# delete cache from previous trials
import shutil, os
for d in ["cache_pretrain", "cache_sft"]:
#for d in ["cache_pretrain"]:
    p = (LOCAL_ROOT / d)
    if p.exists():
        print("Removing", p)
        shutil.rmtree(p)
print("OK")


Removing cache_sft
OK


In [4]:
#!python -m axolotl.cli.preprocess "axolotl_configs/config_stage1_pretrain.yaml"

In [5]:
#!python -m axolotl.cli.preprocess "/content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml"
#!axolotl preprocess /content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml \
#  --debug-num-examples 0
#!python -m axolotl.cli.preprocess "/content/ft_vs_rag_work/axolotl_configs/config_stage2_sft.yaml"
!python -m axolotl.cli.preprocess "axolotl_configs/config_stage2_sft.yaml"


[2025-11-15 15:53:58,867] [WARNING] [axolotl.utils.schemas.model] `trust_remote_code` is set to true. Please make sure that you reviewed the remote code/model.
[2025-11-15 15:53:59,298] [INFO] [axolotl.cli.config] config:
{
  "activation_offloading": false,
  "adapter": "qlora",
  "attn_implementation": "flash_attention_2",
  "axolotl_config_path": "axolotl_configs/config_stage2_sft.yaml",
  "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
  "base_model_config": "mistralai/Mistral-7B-Instruct-v0.2",
  "batch_size": 48,
  "bf16": true,
  "capabilities": {
    "bf16": true,
    "compute_capability": "sm_90",
    "fp8": false,
    "n_gpu": 1,
    "n_node": 1
  },
  "context_parallel_size": 1,
  "dataloader_num_workers": 1,
  "dataloader_pin_memory": true,
  "dataloader_prefetch_factor": 256,
  "dataset_prepared_path": "cache_sft",
  "dataset_processes": 96,
  "datasets": [
    {
      "message_property_mappings": {
        "content": "content",
        "role": "role"
      },
      "pa

In [6]:
# Check first few lines of datasets to ensure schemas are right
!head -n 2 data/docs_for_pretrain.jsonl
!head -n 2 data/docs_qa_with_context_ALPACA.jsonl


head: cannot open 'data/docs_for_pretrain.jsonl' for reading: No such file or directory
{"instruction": "Answer the question. Use the provided context if it helps.", "input": "<CONTEXT>\nArthur magazine was a bi-monthly periodical that was founded in October 2002, by publisher Laris Kreslins and editor Jay Babcock. It received favorable attention from other periodicals such as \"L.A. Weekly\", \"Print\", \"Punk Planet\" and \"Rolling Stone\". \"Arthur\" featured photography and artwork from Spike Jonze, Art Spiegelman, Susannah Breslin, Gary Panter and Godspeed You! Black Emperor. Arthur's regular columnists included Byron Coley, Thurston Moore, Daniel Pinchbeck, Paul Cullum, Douglas Rushkoff, and T-Model Ford.\n\nThe Ladies' Magazine, an early magazine for women, was first published in 1828 in Boston, Massachusetts. Also known as \"Ladies' Magazine and Literary Gazette\" and later as \"American Ladies Magazine\", it was designed to be American, and named to separate itself from the \"

# Cell 7 — Train Stage-1 (live log)

In [7]:
!accelerate launch \
  --num_processes 3 \
  --num_machines 1 \
  --mixed_precision bf16 \
  -m axolotl.cli.train axolotl_configs/config_stage1_pretrain.yaml


The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2025-11-15 15:54:45,055] [WARNING] [axolotl.utils.schemas.validation] Setting `pretraining_dataset` without explicitly setting `streaming: true` is deprecated. In a future release, streaming will not be automatically enabled when using pretraining_dataset. Please explicitly set `streaming: true` in your configuration to maintain current behavior.
[2025-11-15 15:54:45,056] [WARNING] [axolotl.utils.schemas.model] `trust_remote_code` is set to true. Please make sure that you reviewed the remote code/model.
[2025-11-15 15:54:45,671] [INFO] [axolotl.cli.config] config:
{
  "accelerator_config": {
    "dispatch_batches": false,
    

# Cell 8 — Save Stage-1 artifacts to Drive

# Cell 9 — Train Stage-2 (live log)

In [ ]:
!accelerate launch \
  --num_processes 3 \
  --num_machines 1 \
  --mixed_precision bf16 \
  -m axolotl.cli.train axolotl_configs/config_stage2_sft.yaml


The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2025-11-15 23:10:52,631] [WARNING] [axolotl.utils.schemas.model] `trust_remote_code` is set to true. Please make sure that you reviewed the remote code/model.
[2025-11-15 23:10:52,944] [INFO] [axolotl.cli.config] config:
{
  "activation_offloading": false,
  "adapter": "qlora",
  "attn_implementation": "flash_attention_2",
  "axolotl_config_path": "axolotl_configs/config_stage2_sft.yaml",
  "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
  "base_model_config": "mistralai/Mistral-7B-Instruct-v0.2",
  "batch_size": 144,
  "bf16": true,
  "capabilities": {
    "bf16": true,
    "compute_capability": "sm_90",
    "fp8": false,

# Cell 10 — Save Stage-2 artifacts to Drive

# Cell 11 — Quick sanity inference (LoRA adapter from Stage-2)

In [49]:
# ==== פרמטרים שאת משנה לפי הסביבה שלך ====
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # או מה שבחרת
ADAPTER_PATH  = "/content/outputs/mistral_stage1"  # TODO: update path
USE_4BIT      = False  # True אם את רוצה מצב B (ללא offload). אחרת מצב A.
OFFLOAD_DIR   = "/content/offload"  # מצב A בלבד
DTYPE         = "bfloat16"  # "float16" אם ה-GPU לא תומך BF16 (למשל T4)

# ==== התקנות נדרשות בסביבת קולאב/דוקר (אם עוד לא מותקן) ====
# !pip install -q transformers accelerate bitsandbytes peft

import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import PeftModel

# יצירת תיקיית offload אם נשתמש במצב A
if not USE_4BIT:
    os.makedirs(OFFLOAD_DIR, exist_ok=True)

# בחירת dtype
dtype = torch.bfloat16 if DTYPE == "bfloat16" and torch.cuda.is_bf16_supported() else torch.float16

# טוענים tokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token  # להבטיח padding חוקי

# ===== מצב A: device_map="auto" עם offload לדיסק =====
if not USE_4BIT:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        offload_folder=OFFLOAD_DIR,     # <-- זה פותר את ה-ValueError
        torch_dtype=dtype,
    )

# ===== מצב B: טעינה ב-4bit (לרוב מונעת צורך ב-disk) =====
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        torch_dtype=dtype,
    )

# אם יש לך LoRA/PEFT – טעני את האדפטור מעל הבסיס:
if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    # מעדכנים את המודל עם האדפטור. נשארים עם אותו device_map/offload כפי שהוטען לבסיס.
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()

# ===== sanity check קצר =====
prompt = "Write a short paragraph on the advantages of RAG over Fine-Tuning."
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tok.eos_token_id,
    )

print(tok.decode(out_ids[0], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Write a short paragraph on the advantages of RAG over Fine-Tuning.

RAG (Replacing All Tokens) is a promising approach in the field of text generation, which holds several advantages over fine-tuning. Unlike fine-tuning, where the model is trained on a specific dataset with a few labeled examples, RAG enables the model to generate new text by replacing all tokens in the input sequence with new tokens generated from the model itself. This capability makes RAG more versatile and adaptable to various text generation tasks, as it doesn't require extensive labeled data for each new task. Moreover, RAG's generation process is more controllable as the model can


In [7]:
import os
import tarfile
from google.colab import files

output_folder = "/content/outputs/mistral_stage1"
exclude_subfolder = "checkpoint-20"
archive_name = "mistral_stage1_no_checkpoint-20.tar.gz"

# Create the tar archive excluding the specified subfolder
# We change directory to the parent of the output_folder before archiving
# so that the archive does not contain the full path /content/outputs/mistral_stage1
# but rather its contents directly.
with tarfile.open(archive_name, "w:gz") as tar:
    for root, dirs, files_in_dir in os.walk(output_folder):
        # Calculate the path relative to the output_folder
        relative_root = os.path.relpath(root, output_folder)

        # Skip the excluded subfolder itself
        if relative_root == exclude_subfolder:
            # Clear dirs so os.walk doesn't descend into it
            del dirs[:]
            continue

        # Add current directory to archive if not the root of the excluded subfolder
        if relative_root != "." and relative_root != exclude_subfolder:
            tar.add(root, arcname=relative_root)

        # Filter out the excluded subfolder from the list of directories to traverse
        if exclude_subfolder in dirs:
            dirs.remove(exclude_subfolder)

        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # Ensure the archive name is relative to the output_folder
            arcname = os.path.join(relative_root, file)
            tar.add(file_path, arcname=arcname)

print(f"Archive '{archive_name}' created successfully in /content/.")
print(f"You can download it using the Colab file browser or by running: files.download('{archive_name}')")


Archive 'mistral_stage1_no_checkpoint-20.tar.gz' created successfully in /content/.
You can download it using the Colab file browser or by running: files.download('mistral_stage1_no_checkpoint-20.tar.gz')


In [ ]:
import os
import tarfile
from google.colab import files

output_folder = "outputs/mistral_stage2"
exclude_subfolder = "checkpoint-595"
archive_name = "mistral_stage1_no_checkpoint-20.tar.gz"

# Create the tar archive excluding the specified subfolder
# We change directory to the parent of the output_folder before archiving
# so that the archive does not contain the full path /content/outputs/mistral_stage1
# but rather its contents directly.
with tarfile.open(archive_name, "w:gz") as tar:
    for root, dirs, files_in_dir in os.walk(output_folder):
        # Calculate the path relative to the output_folder
        relative_root = os.path.relpath(root, output_folder)
        
        # Skip the excluded subfolder itself
        if relative_root == exclude_subfolder:
            # Clear dirs so os.walk doesn't descend into it
            del dirs[:]
            continue

        # Add current directory to archive if not the root of the excluded subfolder
        if relative_root != "." and relative_root != exclude_subfolder:
            tar.add(root, arcname=relative_root)

        # Filter out the excluded subfolder from the list of directories to traverse
        if exclude_subfolder in dirs:
            dirs.remove(exclude_subfolder)

        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # Ensure the archive name is relative to the output_folder
            arcname = os.path.join(relative_root, file)
            tar.add(file_path, arcname=arcname)

print(f"Archive '{archive_name}' created successfully in /content/.")
print(f"You can download it using the Colab file browser or by running: files.download('{archive_name}')")


In [ ]:
import os
import tarfile
from google.colab import files

output_folder = "outputs/mistral_stage1"
exclude_subfolder = "checkpoint-2763"
archive_name = "mistral_stage1_no_checkpoint-20.tar.gz"

# Create the tar archive excluding the specified subfolder
# We change directory to the parent of the output_folder before archiving
# so that the archive does not contain the full path /content/outputs/mistral_stage1
# but rather its contents directly.
with tarfile.open(archive_name, "w:gz") as tar:
    for root, dirs, files_in_dir in os.walk(output_folder):
        # Calculate the path relative to the output_folder
        relative_root = os.path.relpath(root, output_folder)
        
        # Skip the excluded subfolder itself
        if relative_root == exclude_subfolder:
            # Clear dirs so os.walk doesn't descend into it
            del dirs[:]
            continue

        # Add current directory to archive if not the root of the excluded subfolder
        if relative_root != "." and relative_root != exclude_subfolder:
            tar.add(root, arcname=relative_root)

        # Filter out the excluded subfolder from the list of directories to traverse
        if exclude_subfolder in dirs:
            dirs.remove(exclude_subfolder)

        for file in files_in_dir:
            file_path = os.path.join(root, file)
            # Ensure the archive name is relative to the output_folder
            arcname = os.path.join(relative_root, file)
            tar.add(file_path, arcname=arcname)

print(f"Archive '{archive_name}' created successfully in /content/.")
print(f"You can download it using the Colab file browser or by running: files.download('{archive_name}')")
